In [1]:
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
from data import UncertainDataset

In [3]:
from pathlib import Path

plots = Path("plots_mean_spread")

In [4]:
from matplotlib import pyplot as plt

def plot_ensemble_mean_spread(mean: xr.Dataset, spread: xr.Dataset, plot: Path, sample: slice = slice(None)):
    fig, ax = plt.subplots()
    
    months = sorted(set(pd.to_datetime(mean.valid_time).month.values))
    years = sorted(set(pd.to_datetime(mean.valid_time).year.values))
    
    colours = [
        "tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", 
        "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan", 
        "gold", "teal"
    ]
    
    ax.set_title(mean.attrs.get("long_name", "Ensemble Mean vs Spread"))
    ax.set_xlabel(f"Mean [{mean.attrs.get('units', '')}]")
    ax.set_ylabel(f"Spread [{spread.attrs.get('units', '')}]")
    
    for year in years:
        for month in months:
            colour = colours[(month - 1) % 12]
            
            mean_values = mean.sel(valid_time=(mean.valid_time.dt.year == year) & (mean.valid_time.dt.month == month)).values.flatten()[sample]
            spread_values = spread.sel(valid_time=(spread.valid_time.dt.year == year) & (spread.valid_time.dt.month == month)).values.flatten()[sample]
            
            ax.scatter(
                mean_values,
                spread_values,
                s=2, label=f"{year}/{month:02d}", edgecolor="none", marker=".", c=colour,
            )
    
    fig.legend(loc="upper right", markerscale=10)
    
    plot.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot)
    
    plt.close(fig)


In [5]:
from tqdm import tqdm

In [ ]:
from data import ds_single

for key in tqdm(ds_single.mean):
    plot_ensemble_mean_spread(ds_single.mean[key], ds_single.spread[key], plots / "single" / f"mean_spread_{ds_single.mean[key].attrs.get('GRIB_shortName')}.png", slice(0, None, 1000))

In [6]:
from data import ds_pressure

for key in tqdm(ds_pressure.mean):
    for pressure in ds_pressure.mean[key].pressure_level.values:
        plot_ensemble_mean_spread(
            ds_pressure.mean[key].sel(pressure_level=pressure),
            ds_pressure.spread[key].sel(pressure_level=pressure),
            plots / "pressure" / f"mean_spread_{ds_pressure.mean[key].attrs.get('GRIB_shortName')}-{pressure}hPa.png",
            slice(0, None, 1000),
        )

100%|██████████| 16/16 [1:03:08<00:00, 236.76s/it]
